In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum, max, when, weekofyear, date_format, year, month, dayofmonth, lit
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window


@dp.materialized_view(
    name="dev_emp.silver.employee_day_behaviour",
    comment="Silver layer: Employee daily behavior metrics with time-based aggregations and anomaly coefficients"
)
def employee_day_behaviour():
    """
    Calculate employee behavior metrics aggregated by day, hour, and daytime.
    Includes messages per day/week, hourly patterns, and anomaly detection coefficients.
    """
    # Read from upstream emp_item table in batch mode (required for window functions)
    df = spark.read.table("emp_item")
    
    # Parse ITEM_DATE and extract date/time components
    df_parsed = df.withColumn("parsed_date", F.to_timestamp(col("ITEM_DATE"), "dd-MMM-yy hh:mm:ss a")) \
        .withColumn("date", F.to_date(col("parsed_date"))) \
        .withColumn("hour", F.hour(col("parsed_date"))) \
        .withColumn("account_id", col("ACCOUNT_ID"))
    
    # Count messages per hour for each account
    hourly_counts = df_parsed.groupBy('date', 'hour', 'account_id') \
        .agg(F.count("*").alias("msg_per_hr")) \
        .withColumn("Interval", F.concat(F.lpad(col("hour"), 2, "0"), F.lit(":00-"), F.lpad((col("hour") + 1) % 24, 2, "0"), F.lit(":00")))

    # Day calculation
    day = hourly_counts.groupBy('date', 'Interval', 'hour', 'account_id', 'msg_per_hr').count() \
        .withColumn('messages_per_day',
                    sum("msg_per_hr").over(Window.partitionBy("date", "account_id").orderBy('hour')).cast(
                        IntegerType())) \
        .withColumn("week", weekofyear(col('date'))) \
        .withColumn("dayname", date_format(col('date'), "EEEE")) \
        .withColumn("hour_interval", col('Interval')) \
        .withColumn("daytime", when((col('hour') >= 6) & (col('hour') < 12), 'Morning')
                                .when((col('hour') >= 12) & (col('hour') <= 17), 'Afternoon')
                                .when((col('hour') > 17) & (col('hour') <= 23), 'Evening')
                                .otherwise('Night'))


    # Max messages per day calculation
    maxi = day.withColumn('max_hourly_messages_per_day',
                          max('msg_per_hr').over(Window.partitionBy("date", "account_id")).cast(IntegerType()))


    # Week calculation
    week = maxi.withColumn('messages_per_week',
                           sum('messages_per_day').over(Window.partitionBy("week", "account_id")).cast(
                               IntegerType()))


    # Daytime calculation
    daytime = week.withColumn('messages_per_daytime', sum('msg_per_hr').over(
        Window.partitionBy("date", "daytime", "account_id").orderBy("daytime")).cast(IntegerType()))
    
    defaultcoefficients = '''{
  'bm_messages_per_hour_weight': 0.05,
  'bm_messages_per_daytime_weight': 0.05,
  'bm_messages_per_day_weight': 0.05,
  'bm_max_hourly_messages_per_day_weight': 0.05,
  'bm_messages_per_week_weight': 0.05,
  'dt_messages_per_specific_daytime_weight': {'Morning': 0.05, 'Afternoon':0.05,'Evening':0.05,'Night':0.05},
  'dn_messages_per_specific_day_weight': {'Monday':0.05,'Tuesday': 0.05, 'Wednesday':0.05,'Thursday':0.05,'Friday':0.05,'Saturday':0.05,'Sunday':0.05},
  'dn_max_hourly_messages_per_specific_day_weight': {'Monday':0.05,'Tuesday': 0.05, 'Wednesday':0.05,'Thursday':0.05,'Friday':0.05,'Saturday':0.05,'Sunday':0.05}}'''


    # Active coefficients calculation
    df_active = daytime.withColumn("year", year(col('date').cast("timestamp"))).withColumn('month', month(
        col('date').cast('timestamp'))).withColumn("day", dayofmonth(col('date').cast('timestamp'))) \
        .withColumnRenamed("msg_per_hr", "messages_per_hour") \
        .withColumn('anomaly_update_coefficients', lit(defaultcoefficients)) \
        .withColumn('default_update_coefficients', lit(defaultcoefficients))
    
    return df_active